# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description
print(f"{getattr(metadata, 'name', 'No name found')}: {getattr(metadata, 'description', 'No description found')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
record_sets = metadata.recordSets
print("Record sets in dataset (with their @id):")
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    print(f"- @id: {rs_id}, name: {getattr(rs, 'name', None)}")

# Example: List fields for the first record set
if len(record_sets) > 0:
    sample_record_set = record_sets[0]
    print(f"\nFields in record set '@id: {getattr(sample_record_set, '@id')}'")
    for field in getattr(sample_record_set, 'fields', []):
        print(f"  - @id: {getattr(field, '@id', None)}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'dataType', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set @id's
record_set_ids = [getattr(rs, '@id') for rs in metadata.recordSets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records from '{record_set_id}'")
    else:
        print(f"Warning: No records found for record set '@id: {record_set_id}'")

# Pick one main record set to inspect (the first with data)
main_record_set_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes:
        main_record_set_id = rs_id
        break

if main_record_set_id:
    print(f"\nColumns for main record set ({main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No tabular record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing fields, and grouping. All column references use their respective field/column `@id`.

In [ ]:
# --- Choose a numeric field by its @id ---
# We'll select the first numeric field found in the selected record set.
numeric_field_id = None
group_field_id = None

fields = []
for rs in metadata.recordSets:
    if getattr(rs, '@id') == main_record_set_id:
        fields = getattr(rs, 'fields', [])

# Identify the first numeric field (Float or Integer)
for field in fields:
    data_type = getattr(field, 'dataType', None)
    if data_type in ('Float', 'Number', 'Integer'):
        numeric_field_id = getattr(field, '@id')
        break

# Identify a non-numeric field for grouping (e.g., protein description)
for field in fields:
    if getattr(field, 'dataType', None) == 'Text':
        group_field_id = getattr(field, '@id')
        break

df = dataframes[main_record_set_id]

if numeric_field_id and numeric_field_id in df.columns:
    # Clean/convert the column if necessary
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f}: {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a non-numeric field, if found
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped average '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and main_record_set_id in dataframes:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=40, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        # Plot average of numeric by group field (top 10 groups)
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10, 4))
        sns.barplot(x=grouped.index, y=grouped.values)
        plt.xticks(rotation=45, ha='right')
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.title(f'Mean {numeric_field_id} by {group_field_id} (top 10)')
        plt.show()

## 6. Conclusion
Summarized above, we loaded the FAIR^2-compliant dataset, explored the available record sets and fields, filtered and normalized numeric data, and generated visualizations for key numeric fields. Further biological interpretation depends on the semantics of each field (`@id`).